# Advanced Prompt Evaluation with Gemini (Criterion-based)

This notebook implements a sophisticated evaluation pipeline that generates specific evaluation criteria for each task, leading to more accurate model reviews.

### 1. Setup and Initialization
We import essential libraries and initialize the Google GenAI client using the `gemini-2.5-flash` model.

In [1]:
import os
import json
import re
import ast
from statistics import mean
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.5-flash"

### 2. Advanced Structured Schemas
In this version, we extend the `Task` schema to include `solution_criteria`. This allows the model to generate its own "rubric" for each task, which will be used later during the evaluation phase.

In [2]:
# Define Structured Output Schemas
class TaskWithCriteria(BaseModel):
    task: str
    format: str # 'python', 'json', or 'regex'
    solution_criteria: str # Specific requirements for the reviewer

class DatasetWithCriteria(BaseModel):
    tasks: list[TaskWithCriteria]

class Evaluation(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: float

### 3. Generation with Criteria
The `generate_dataset` function now asks Gemini to define what a "correct" solution looks like for each generated task. This rubic-based approach significantly improves the consistency of the final scores.

In [3]:
def generate_dataset(count=3):
    prompt = f"""
    Generate an evaluation dataset for AWS-related tasks.
    For each task, provide specific 'solution_criteria' that a reviewer should use to grade the solution.
    Generate {count} unique tasks.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DatasetWithCriteria
        )
    )
    return response.parsed.tasks

def run_prompt(test_case):
    prompt = f"""
    Solve the following task:
    {test_case['task']}
    
    * Respond ONLY with the requested {test_case['format']} code.
    * Do not include markdown blocks or commentary.
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response.text.strip()

### 4. Criterion-based Model Grading
The `grade_by_model` function now passes the specific `solution_criteria` into the system instructions. This ensures the model judges the code based on the pre-defined requirements rather than generic assumptions.

In [4]:
def validate_syntax(text, format):
    try:
        if format == "json":
            json.loads(text)
        elif format == "python":
            ast.parse(text)
        elif format == "regex":
            re.compile(text)
        return 10
    except:
        return 0

def grade_by_model(test_case, output):
    prompt = f"""
    Evaluate this AWS-related solution.
    Task: {test_case['task']}
    Criteria: {test_case['solution_criteria']}
    Solution: {output}
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are an expert AWS code reviewer. Use the provided criteria to grade rigorously.",
            response_mime_type="application/json",
            response_schema=Evaluation
        )
    )
    return response.parsed

### 5. Final Orchestration
The final block executes the end-to-end flow and outputs a detailed summary of each test case, including the output, the score, and the model's reasoning based on the generated criteria.

In [5]:
def run_eval():
    print("Generating dataset with criteria...")
    dataset = generate_dataset()
    
    # Save dataset to root data/ folder
    os.makedirs("../../data", exist_ok=True)
    with open("../../data/dataset_complete.json", "w") as f:
        json.dump([d.model_dump() for d in dataset], f, indent=2)
    print("Dataset saved to ../../data/dataset_complete.json")
    
    results = []
    
    for case in dataset:
        case_dict = case.model_dump()
        print(f"Running test case: {case_dict['task'][:50]}...")
        
        output = run_prompt(case_dict)
        syntax_score = validate_syntax(output, case_dict['format'])
        model_eval = grade_by_model(case_dict, output)
        
        avg_score = (syntax_score + model_eval.score) / 2
        
        results.append({
            "task": case_dict['task'],
            "output": output,
            "criteria": case_dict['solution_criteria'],
            "score": avg_score,
            "reasoning": model_eval.reasoning
        })
    
    print(f"\nEvaluation Complete. Average Score: {mean([r['score'] for r in results]):.2f}")
    
    # Save results to root data/ folder
    with open("../../data/results_complete.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Results saved to ../../data/results_complete.json")
    
    return results

results = run_eval()
import json
print(json.dumps(results, indent=2))

Generating dataset with criteria...
Dataset saved to ../../data/dataset_complete.json
Running test case: Create an S3 bucket named 'my-unique-test-bucket-1...
Running test case: Launch an EC2 t2.micro instance using the latest A...
Running test case: Create a Python 3.9 Lambda function named 'MyHello...

Evaluation Complete. Average Score: 7.13
Results saved to ../../data/results_complete.json
[
  {
    "task": "Create an S3 bucket named 'my-unique-test-bucket-12345' (replace 12345 with a unique identifier) in the 'us-east-1' region. The bucket must have S3 Object Versioning enabled, Block All Public Access enabled, and Default Encryption (SSE-S3) configured.",
    "output": "BUCKET_NAME=\"my-unique-test-bucket-$(date +%s)\"\naws s3api create-bucket --bucket \"${BUCKET_NAME}\" --region us-east-1\naws s3api put-bucket-versioning --bucket \"${BUCKET_NAME}\" --versioning-configuration Status=Enabled\naws s3api put-public-access-block --bucket \"${BUCKET_NAME}\" --public-access-block-confi